# **단백질 언어 모델(Protein Language Model)을 이용한 항체 설계 입문**


#### 작성: Dhuvi Karthikeyan, Aaron Menezes


이 DeepChem 튜토리얼은 단백질 언어 모델(protein language model)을 통한 항체(antibody) 설계의 간단한 입문서로 만들어졌습니다. 항체는 면역글로불린(immunoglobulin)이라고도 불리는 면역 단백질로, 체내에서 자연적으로 생성되어 바이러스나 다른 병원체에 결합·무력화합니다. 항체는 귀중한 치료제로, 암에 대한 면역관문억제제(immune checkpoint inhibitor)나 급성 바이러스 감염에 대한 중화항체(neutralizing antibody)로서 생명을 구합니다. 또한 병원 밖에서도, 임의의 분자 표적에 결합·부착하는 능력으로 잘 알려져 있어 기초 과학과 산업 생화학 시설에서 유용하게 쓰입니다.

이 튜토리얼은 면역계라는 더 넓은 맥락에서 항체의 구조와 기능을 이해하는 데 필요한 핵심 면역학 개념을 빠르게 개관하는 것을 목표로 합니다. 대형 언어 모델(LLM)에 어느 정도 익숙하다는 것을 전제로 합니다. 관련해서는 저희의 다른 [튜토리얼](https://colab.research.google.com/drive/13eXPgZpzTOL3c_S7OM7uR6btP8ZWw7zn)도 참고하세요. 간결함을 위해, 핵심이 아닌 주제는 가능한 한 외부 자료 링크로 대신합니다. 함께 따라오며 면역계와, 안내된(guided) 항체 설계를 위한 단백질 언어 모델에 대해 더 알아봅시다.


**참고:** 이 튜토리얼은 Hie 등이 쓴 2023년 Nature Biotechnology 논문 "Efficient evolution of human antibodies from general protein language models" [[1]](https://www.nature.com/articles/s41587-023-01763-2#Sec37)을 느슨하게 기반으로 합니다. 방법과 데이터를 공개해 주신 저자들께 감사드립니다.

## Colab에서 실행하기

이 노트북은 Google Colab에서 실행하는 것을 권장합니다. 아래 Colab 배지를 클릭하면 바로 열립니다. (Kaggle에서도 열 수 있습니다.)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/isg-yhlee93/lecture/blob/main/Modeling%20Proteins/2_DeepChem_AntibodyTutorial_Simplified.ipynb)
[![Open In Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/kernels/welcome?src=https://github.com/isg-yhlee93/lecture/blob/main/Modeling%20Proteins/2_DeepChem_AntibodyTutorial_Simplified.ipynb)


In [ ]:
# 필요한 패키지를 설치합니다 (Colab에는 보통 이미 설치되어 있습니다).
!pip install -qq transformers

# 불필요한 경고 메시지 끄기
import warnings
warnings.filterwarnings('ignore')


## 1. 면역학 기초 (Immunology 101)



### 1.1 면역계(immune system)란 무엇인가?

소화계나 심혈관계 같은 다른 신체 계통과 마찬가지로, 면역계도 근본적으로는 특정한 [항상성(homeostasis)](https://en.wikipedia.org//wiki/Homeostasis) 기능을 함께 수행하기 위해 협력하는 특수화된 세포들의 집합입니다. 면역계는 특히 바이러스나 박테리아 같은 외부 세계의 기회감염 위협으로부터, 우리 몸의 방대한 자원(지질, 탄수화물, 효소, 단백질)을 보호하는 역할을 합니다. 면역계의 매우 단순하지만 강력한 구성 요소 하나가 피부인데, 피부는 밖의 것은 밖에, 안의 것은 안에 머물게 합니다. 그런데 베이거나 긁히면, 또는 기침하는 사람 옆을 지나며 비말을 들이마시면 어떻게 될까요? 다행히 면역계에는 우리 혈액과 림프절을 순찰하는 고도로 특수화된 백혈구(white blood cell) 부대가 있어, 이런 경우와 외부 물질이 몸에 들어올 수 있는 수많은 다른 경우에 대해 다층적 반응을 펼칠 준비가 되어 있습니다. 이런 백혈구를 통해 면역계는 핵심 기능인 **자기-비자기 구별(self-nonself discrimination)**, 즉 건강한 사람에게서 유래한 분자와 충분히 다른 분자를 정확히 식별하는 일을 해냅니다.

자기-비자기 구별이라는 복잡한 문제를 더 알고 싶고, 면역계의 구성과 기능 뒤에 있는 이론을 음미하고 싶다면 다음 자료들을 추천합니다:

* A Theory of Self-Nonself Discrimination [[2]](https://www.science.org/doi/10.1126/science.169.3950.1042)
* The common sense of the self-nonself discrimination, 2005 [[3]](https://link.springer.com/article/10.1007/s00281-005-0199-1)
* Conceptual aspects of self and nonself discrimination, 2011 [[4]](https://www.tandfonline.com/doi/epdf/10.4161/self.2.1.15094?needAccess=true)
* Self-Nonself Discrimination due to Immunological Nonlinearities: the Analysis of a Series of Models by Numerical Methods, 1987 [[5]](https://academic.oup.com/imammb/article/4/1/1/875656)
* A biological context for the self-nonself discrimination and the regulation of effector class by the immune system, 2005 [[6]](https://link.springer.com/article/10.1385/IR:31:2:133)



### **1.2 선천 면역계 vs. 적응 면역계 (Innate vs. Adaptive)**

놀랍게도 이 [백혈구](https://en.wikipedia.org/wiki/White_blood_cell)들은 독립적으로 작동하며, 국소적 맥락(조직 미세환경)에 더하거나 빼면서 완전히 분산된 방식으로 응집된 반응을 조율합니다. 개별 백혈구는 일반적인 [사이토카인(cytokine)](https://en.wikipedia.org/wiki/Cytokine)·[케모카인(chemokine)](https://en.wikipedia.org/wiki/Chemokine) 표면 수용체를 통해 환경과 상호작용하며, 이를 통해 국소 환경의 농도 기울기를 감지합니다. 이것이 면역세포가 감염/상처 치유 부위로 이동(home)하고 이웃 세포에 염증 촉진(pro-inflammatory) 또는 염증 억제(anti-inflammatory) 신호를 보내는 주요 축입니다. 또한 거의 모든 면역세포는 비자기(non-self) 신호에 의한 수용체:리간드(receptor:ligand) 자극을 통해 어느 정도의 활성화(activation)를 필요로 합니다. 이 수용체가 얼마나 특이적인지는 면역계의 두 갈래, 즉 선천 면역계와 적응 면역계 사이의 구분을 잘 보여줍니다.

#### 1.2.1 선천 면역계(Innate Immune System)

[선천 면역계](https://en.wikipedia.org/wiki/Innate_immune_system)는 신체의 1차 방어선으로, 병원체 연관 분자 패턴 또는 손상 연관 분자 패턴([PAMP](https://en.wikipedia.org/wiki/Pathogen-associated_molecular_pattern) 또는 [DAMP](https://en.wikipedia.org/wiki/Damage-associated_molecular_pattern))으로 알려진 광범위한 비자기 신호를 인식하는 다양한 세포(와 단백질)를 포함합니다. 선천 면역계의 효과기 세포(effector cell)는 약 20종의 패턴 인식 수용체(PRR)로 덮여 있어, 광범위한 신호(단일가닥 DNA, LPS, 그 밖의 병원체 활동 신호)를 인식합니다. 1차 대응자로서 선천 면역계는 봉쇄(containment)를 담당하여, 문제를 일으키는 신호를 집어삼키거나(engulf) 차단하고 위협을 파괴하려 합니다. 마지막으로 이 세포들은 동시에 경보를 울려, 발열을 일으키고 부종을 시작해 더 많은 면역세포를 끌어들이는 등, 국소 부위를 위협 탐지·감염 예방의 고조된 상태로 만듭니다.

#### 1.2.2 적응 면역계(Adaptive Immune System)

반면 [적응 면역계](https://en.wikipedia.org/wiki/Adaptive_immune_system)는 반응이 느리고, 흔히 선천 면역세포의 활성화에 의해 동원됩니다. 광범위한 PRR 대신, 적응 면역계는 수천만 개의 [체세포 재배열(somatic recombination)](https://en.wikipedia.org/wiki/Somatic_recombination)되고 확률적으로 생성된 고도로 특이적인 적응 면역 수용체(AIR)에 의존하며, 그 형태 상보성(shape complementarity) 덕분에 자신과 상보적인 분자 패턴인 에피토프(epitope)를 식별할 수 있습니다. 선천 면역계와 적응 면역계 사이에는 후자가 지속적인 기억 반응(memory response)을 형성하는 능력 등 더 많은 차이가 있으며, 이는 여기 [[7]](https://www.ncbi.nlm.nih.gov/books/NBK27090/)에서 더 자세히 소개됩니다. 다만 위협 탐지, 메시지 전달, 지원 요청, 적응 면역세포의 이동, 그리고 이 세포들의 활성화/증식을 거쳐 병원체를 완전히 제거하는 [동역학](https://www.nature.com/articles/nri700) [[8]]은, 선천·적응 면역계 양쪽의 독립적이고 비동기적인 작동을 통해 나타난다는 점이 중요합니다. 선천 면역계와 적응 면역계의 또 다른 핵심 차이는 강력한 기억 반응의 형성입니다. 놀랍게도 진화는 이 시스템이 과거의 병원체 노출을 각인하도록 이끌어, 같은 신호에 재노출되면 소수의 기억 세포가 활성화되고 항원 특이적 적응 면역세포가 빠르게 증식하여 항원원(antigen-source)을 제거하게 합니다 [[9]](https://asm.org/articles/2023/may/understanding-immunological-memory).


```algorithmic
적응 면역 알고리즘

1. 미성숙 전구세포(progenitor cell)가 체세포 재배열을 거쳐 적응 면역 수용체(AIR)를 확률적으로 생성합니다.
2. 자기 관용(self-tolerance)을 보장하는 자기 선택(self-selecting) 과정이, 혈액으로 방출되기 전에 자기(self)에 너무 강하게 반응하는 세포를 제거합니다.
3. 말초에서, 미경험(naive, 항원을 만난 적 없는) 세포가 AIR를 통해 항원과 상호작용하며 자신의 동족(cognate) 에피토프를 찾습니다.
4. 충분히 강한 에피토프 반응을 인식하면, 세포가 빠르게 분열하여 막대한 수와 특수화된 효과기 기능으로 문제의 항원을 압도합니다.
5. 위협이 사라지면, 활성화 신호 없이 세포가 죽으면서 이 확장된 집단이 수축하고, 지속적인 기억 세포 풀(pool)이 남습니다.
6. 기억 풀은 유지되다가 항원이 재도입되면 효과기 상태로 재활성화됩니다.
```
### 1.2.3 적응 면역계의 효과기 세포(Effector Cells)
적응 면역 반응에는 두 가지 주요 효과기 세포가 있습니다: [T세포(T-cell)](https://en.wikipedia.org/wiki/T_cell)와 [B세포(B-cell)](https://en.wikipedia.org/wiki/B_cell). 체내의 T세포와 B세포 집단은 각각의 적응 면역 수용체, 즉 T세포 수용체(TCR)와 B세포 수용체(BCR)로 규정됩니다. 두 집단 모두 수용체의 레퍼토리(repertoire)로 생각할 수 있으며, 그 레퍼토리가 인식하는 위협에 대해 보호를 제공합니다. T세포는 TCR을 사용해 세포 표면에 제시된 재활용 단백질 조각을 살펴봄으로써 우리 몸의 세포 내(intracellular) 구획을 검사하여 [세포성 면역(cellular immunity)](https://en.wikipedia.org/wiki/Cell-mediated_immunity)을 유지합니다(T세포 매개 면역은 여기 [[10]](https://www.ncbi.nlm.nih.gov/books/NBK10762/)에서 더 읽어보세요). 반면 B세포는 BCR을 사용해 세포 외(extracellular) 구획을 살피며, [체액성 면역(humoral immunity)](https://en.wikipedia.org/wiki/Humoral_immunity), 즉 혈액과 혈장에 떠다니는 위협을 중화하는 일을 맡습니다. 적응 면역 반응의 효과기 세포로서 T세포와 B세포는 발달 과정과 단위로서의 작동 방식이 유사하지만, 세포별 구체적 기능은 상당히 다릅니다.

**참고**: 항원(antigen)과 에피토프(epitope)의 유용한 구분은, 항원은 광범위하게 면역 반응을 일으키는 것으로 여러 에피토프를 가질 수 있다는 점입니다. 에피토프는 적응 면역 수용체의 결합 표면인 파라토프(paratope)와 맞물리는 특정 분자 패턴입니다.


![immune_system.png](https://www.creative-diagnostics.com/upload/image/innate%20and%20adaptive%20immunity2.png)

이미지 출처: [Creative Diagnostics](https://www.creative-diagnostics.com/innate-and-adaptive-immunity.htm)


### **1.3 B세포와 항체 (B-Cells and Antibodies)**


B세포(B-림프구)는 골수(bone marrow)에서 기원해서가 아니라, 닭의 특정 기관에서 발견되었기에 그 이름이 붙었습니다 [[11]](https://www.sciencedirect.com/science/article/pii/S0032579119369536?via%3Dihub). 이 세포들은 혈류를 순환하며, 특정 항원을 인식하게 해주는 고유한 B세포 수용체(BCR)를 갖추고 있어 활성화됩니다. 이 과정은 흔히 보조 T세포(helper T-cell)의 추가 자극을 필요로 하는데, 보조 T세포는 비자기성에 대한 추가 검증 단계로서 필수적인 공동자극(co-stimulatory) 신호를 제공합니다.

코로나(COVID) 팬데믹을 거치며, 원하든 원치 않든 우리는 항체라는 개념에 노출되었고 항체가 SARS-CoV-2에 대한 어떤 보호 능력과 연관된다는 것을 알게 되었습니다. 그런데 항체란 무엇이고 어디에서 비롯될까요?

항체(Ab)는 보통 Y자 모양의 단백질로 표현되며, TCR이나 BCR이 에피토프에 결합하는 것과 비슷하게, 자신의 동족 에피토프 표면에 높은 특이성(specificity)과 친화도(affinity)로 결합합니다. 이는 항체가 곧 B세포 수용체의 가용성(soluble) 형태로서, 동족 항원이 있을 때 B세포가 활성화되면 혈액으로 분비되기 때문입니다. 다량의 항체 분비가 B세포의 주요 효과기 기능입니다. 활성화되면 B세포는 분열하고, 딸세포들은 같은 BCR을 물려받으며, 그중 일부는 형질세포(plasma cell)로 분화합니다. 형질세포는 분당 수천 개의 항체를 분비할 수 있는 항체 공장입니다. 이는 특히 항원을 재차 만났을 때 유용한데, 기억 세포가 다량의 항체를 방출하여 우리가 감염 증상을 보이기도 전에 병원체를 중화합니다(대부분의 흔한 백신이 바로 이를 노리고 설계됩니다).

![b_cell_activation.png](https://media.beckman.com/-/media/stock-images/resource-center/images/b_cell_activation-2022-05.jpg?rev=4680b251612b4f0c9afc3280029d0cc7)

이미지 출처: [Beckman](https://www.beckman.com/resources/cell-types/blood-cells/leukocytes/lymphocytes/b-cells)

병원성 기전의 중화는 항체 표지(tagging)가 면역 방어에 유용한 한 가지 방식일 뿐입니다. 항체 표지는 여러 체액성 면역 과정에서 핵심 역할을 합니다:

1. [중화(Neutralization)](https://en.wikipedia.org/wiki/Neutralizing_antibody): 항체가 병원체나 독소의 기능적 부분을 거의 완전히 덮어, 숙주 세포와의 상호작용을 억제함으로써 병원성 기능을 비활성화합니다(예: SARS-CoV-2 표면 당단백질에 결합한 항체는 그 바이러스 입자가 ACE2를 발현하는 세포로 들어가는 능력을 억제합니다).

2. [옵소닌화(Opsonization)](https://en.wikipedia.org/wiki/Antibody_opsonization): 병원체를 부분적으로 덮어, 선천 면역계 세포에 의한 식균작용(phagocytosis)과 혈액에서의 제거 속도를 높입니다.

3. [응집/침전(Agglutination/Precipitation)](https://en.wikipedia.org/wiki/Agglutination_(biology)): 항체는 두 개의 팔(Y의 각 팔)을 가지므로, 교차결합(cross-link)하여 항체-항원 사슬을 형성할 수 있고, 이는 혈장에서 침전되어 이상(aberrant)으로 인식되고 식세포에 의해 제거될 가능성을 높입니다.

4. [보체 활성화(Complement Activation)](https://en.wikipedia.org/wiki/Complement_system): 보체계는 비활성 단백질과 단백질 전구체의 집합으로, 활성화되면 스스로 증폭하며 체액성 면역의 여러 측면을 돕습니다. 항체의 또 다른 기능은, 병원체의 용해(lysis)나 식균으로 끝나는 보체 연쇄반응(complement cascade)을 개시하는 역할입니다.

![antibody_function.png](https://www.agproud.com/ext/resources/PD/images/stories/2021/03/11/031121-pd-weiland-fg1.jpg?t=1695665765&width=610)

이미지 출처: The Immune System: Innate and Adaptive Body Defenses Figure 21.15 ([출처](https://www.agproud.com/articles/35884-understanding-vaccines))

신체의 항체 레퍼토리로 구현되는 B세포 매개 면역의 중요성을 볼 때, BCR 클론의 다양성이 병원체에 대한 효과적인 반응을 펼치는 능력에 결정적 역할을 한다는 것은 분명합니다. 강력한 BCR 레퍼토리의 유지는 면역 반응의 복잡성을 보여줄 뿐 아니라, 이 메커니즘의 모듈성을 활용하여 백신 개발 같은 치료에서 뛰어난 정밀도를 갖는 새로운 클론을 도입할 잠재력도 강조합니다.


### **1.4 항체의 서열, 구조, 기능**

항체의 놀라운 다양성은 체세포 재배열, 즉 감수분열(meiosis) 밖에서 일어나는 DNA 수준의 유전자 재배열을 통해 달성됩니다. AIR 특이적 체세포 재배열은 [V(D)J 재조합](https://en.wikipedia.org/wiki/VDJ_recombination)으로 알려져 있으며 TCR과 BCR 다양성을 모두 생성합니다. V(D)J 재조합 과정에서, 가변(V)·다양성(D)·연결(J) 유전자 분절 집합에서 집합당 하나의 유전자를 뽑아, 약간의 내재적 오류(삽입)와 함께 무작위로 이어 붙여 고유한 항원 결합 부위를 가진 안정적인 BCR을 만듭니다. 또한 B세포에는 항원 특이적 항체의 다양성과 기능적 능력을 더욱 증폭하는 추가 과정이 있습니다. 이를 체세포 과돌연변이(somatic hypermutation, SHM)라고 합니다. 활성화된 B세포가 분열할 때 SHM은 BCR의 가변 영역에 점 돌연변이(point mutation)를 도입하여 각 BCR의 미세한 변이를 만듭니다. 이 딸세포들은 항원 결합을 통해 매개되는 생존 신호를 두고 경쟁하며, 더 강하게 결합하는 것만 살아남습니다.

구조적으로 항체는 동일한 두 개의 경쇄(light chain)와 동일한 두 개의 중쇄(heavy chain)가 이황화 결합(disulfide bond)으로 연결되어 구성됩니다. 각 사슬은 가변 영역(variable region)에 위치한 항원 결합 부위의 형성에 기여합니다. 이 영역 안의 초가변 루프(hypervariable loop), 즉 상보성 결정 영역(CDR)이 항체-항원 상호작용의 특이성과 친화도를 결정합니다. 이 특이성은 해리상수(dissociation constant, Kd)를 사용한 친화도, 그리고 다가(multi-valent) 결합 부위에 대한 친화도인 결합력(avidity)으로 측정됩니다([IgM](https://en.wikipedia.org/wiki/Immunoglobulin_M), [IgA](https://en.wikipedia.org/wiki/Immunoglobulin_A) 참고). 항체 분자는 두 가지 주요 기능 영역으로 나뉩니다:

1. Fab 영역(Fragment, antigen-binding): 경쇄와 중쇄의 가변 영역을 포함하며, 항원 인식과 결합을 담당합니다.
2. Fc 영역(Fragment, crystallizable): 중쇄의 불변 영역(constant region)으로 구성되며, 선천 면역세포 및 보체계와의 상호작용을 매개합니다.

![test_image](https://www.dianova.com/en/wp-content/uploads/sites/3/2021/01/Antikorperstruktur-IgG.jpg)

이미지 출처: Dianova [Antibody Structure](https://www.dianova.com/en/faq/antibody-structure-what-is-a-secondary-antibody/) \\

체세포 과돌연변이 동안의 진화적 선택압을 활용함으로써, B세포 구획은 항체 특이성을 더욱 정교하게 조율하여 알려진 단백질 세계에서 가장 높은 친화도의 상호작용 일부를 만들어내는 강력한 방법을 사용합니다 [[12]](https://www.embopress.org/doi/full/10.1093/emboj/cdg359). 이들의 높은 정밀도와 결합 친화도 덕분에, 항체는 치료제뿐 아니라 상업·연구 응용에서도 널리 채택되어, 유세포분석(flow cytometry), CyTOF, 면역침전(immunoprecipitation) 및 기타 표적 식별 분석에서 용액 속 단백질에 표지를 붙이는 데 쓰입니다.


### **1.5 치료 수단(therapeutic modality)으로서 항체의 현재 패러다임**

임의의 표적에 정밀하고 지속적으로 결합하는 비할 데 없는 능력 덕분에, 원하는 표적에 대한 항체를 확보하는 데 큰 관심이 있어 왔습니다. 이식 거부, 비호지킨 림프종, 암 면역치료를 위한 면역관문억제제, 건선, 다발성 경화증, 크론병 등 다양한 질환에 대한 치료적 용도가 있습니다. 흔한 병원체에 대한 항체는 회복기 환자의 혈청에서 분리하여 특이성을 선별할 수 있지만, 새로운 항체를 확보하는 과정은 훨씬 더 어렵고, 동물에게 항원을 접종한 뒤 항체를 분리하는 과정을 거칩니다. 예를 들어 [항독소혈청(anti-venom)](https://en.wikipedia.org/wiki/Antivenom)은 독에 들어 있는 세포독성 단백질에 대해 동물(보통 말)로부터 얻은 항체 용액입니다. 이 절차는 자원과 노동이 모두 많이 듭니다. 동물에서 항체를 접종·분리한 뒤에도, [표면 플라스몬 공명(surface plasmon resonance)](https://en.wikipedia.org/wiki/Surface_plasmon_resonance)이나 [생체층 간섭법(bio-layer interferometry)](https://en.wikipedia.org/wiki/Bio-layer_interferometry) 같은 반응 화학 기법으로 표적에 대한 반응성을 선별하는 추가 단계가 필요하기 때문입니다. 그래서 *인 실리코(in-silico)* 항체 설계 방법에 큰 관심이 모였습니다. 안내된 진화(guided evolution) 기반 접근 [[1]](https://www.nature.com/articles/s41587-023-01763-2)부터 더 새로운 확산(diffusion) 기반 접근 [[13]](https://arxiv.org/abs/2308.05027)까지 여러 방법이 이 과제에서 상당한 성공을 보였습니다. 이 튜토리얼에서는 전자(안내된 진화)에 경의를 표합니다.


## 2. 코딩해 봅시다! 유도 진화(Directed Evolution)를 통한 항체 설계


### **2.1 개요**

항체 설계 문제를 이해하는 데 필요한 최소한의 배경과 [언어 모델 배경 지식](https://colab.research.google.com/drive/13eXPgZpzTOL3c_S7OM7uR6btP8ZWw7zn#scrollTo=0PMAl8z_HEBr)을 갖추었으니, 아래 그림처럼 유도 진화를 통한 항체 설계로 바로 들어가 봅시다:

\\

![directed_evolution](https://media.springernature.com/full/springer-static/image/art%3A10.1038%2Fs41587-023-01991-6/MediaObjects/41587_2023_1991_Fig1_HTML.png?as=webp)

이미지 출처: Figure 1. [Outeiral et al.](https://www.nature.com/articles/s41587-023-01991-6)


### **2.2 설정/방법론 (Setup/Methodology)**


Hie 등의 연구에서 저자들은 항체 서열에 특화되어 학습된 모델 대신 일반(general) 단백질 언어 모델을 사용하기로 했습니다. 그들은 각각 UniRef50과 UniRef90 [[14]](https://academic.oup.com/bioinformatics/article/23/10/1282/197795?login=true)으로 학습된 ESM-1b와 ESM-1v 모델을 사용합니다. 유도 진화 연구를 위해, 인플루엔자·에볼라바이러스·SARS-CoV-2에 걸친 바이러스 감염과 관련된 7개의 치료용 항체를 선정합니다. 저자들은 항원 결합 영역의 모든 잔기를 다른 모든 잔기로 바꿔보며 서열의 가능도(likelihood)를 계산하는, 단순하고 철저한 돌연변이 스케줄러를 사용합니다. 가능도가 야생형(WT) 서열 이상인 서열은 실험 검증을 위해 남깁니다. 우리의 목적에서는 그렇게까지 철저할 필요가 없으며, 특정 지점에서 상위 k개(top-k) 돌연변이를 취하는 약간 빠른 방법을 사용할 수 있습니다.

Hie 등의 연구에서 영감을 받아, 우리는 pLM(단백질 언어 모델) 기반 유도 진화 과제를, 마스킹 언어 모델링(masked language modeling) 목표로 미리 학습된 pLM에 가려진(masked) 항체 서열을 넣고 마스킹된 아미노산에 대한 토큰 확률을 살펴보는 것으로 단순하게 정의합니다. 정말 이렇게 쉽습니다!

참고로, 과제를 다음 단계로 나눕니다:

```
# pLM 유도 진화를 통한 항체 설계
1. 사전학습된 언어 모델을 선택합니다(모든 도메인에 대해 학습된 것이거나 항체 전용일 수 있음).
2. 돌연변이시킬 항체를 고릅니다.
3. 마스킹할 아미노산을 정합니다*.
4. 토큰화된 서열을 pLM에 넣습니다.
5. 적합도(fitness)를 높이는 휴리스틱에 따라 토큰을 샘플링합니다.
```
*항체 수정은 가변 영역(variable region)에만 집중해야 합니다. 계면(interface)의 아미노산이 친화도를 좌우하기 때문입니다. 불변 영역(constant region)을 수정하면 보체계에서의 항체 효과기 기능에 해롭고, 선천 면역 수용체와의 결합도 방해할 수 있습니다. \\


#### 2.2.1 모델 + 토크나이저 불러오기

이 탐구에서는 초기 항체 언어 모델인 AbLang [[15]](https://academic.oup.com/bioinformaticsadvances/article/2/1/vbac046/6609807?login=true)에 경의를 표합니다. AbLang은 RoBERTa [[16]](https://arxiv.org/abs/1907.11692) 모델을 기반으로 한 마스킹 언어 모델로, [관찰된 항체 공간(observed antibody space, OAS)](https://opig.stats.ox.ac.uk/webapps/oas/oas_paired/) [[17]]의 항체 서열로 사전학습되었습니다. AbLang은 두 모델로 구성되는데, 하나는 중쇄(heavy chain) 서열로, 다른 하나는 경쇄(light chain) 서열로 학습되었습니다. 저자들은 이 모델이 ESM-1b 같은 더 광범위한 단백질 언어 모델보다 유용함을 보여, Hie 등의 결론과는 상반되는 결과를 제시합니다. 중쇄·경쇄 모델은 구조가 동일하며, $d_{model}$=768, 최대 위치 임베딩(max position embedding)=160, 12개의 트랜스포머 블록 층으로 총 약 8,600만 개의 파라미터를 갖습니다.


In [ ]:
from transformers import AutoModel, AutoTokenizer, AutoModelForMaskedLM

# 토크나이저를 가져옵니다.
tokenizer = AutoTokenizer.from_pretrained('qilowoq/AbLang_light')

# 경쇄(light chain) 모델
mlm_light_chain_model = AutoModelForMaskedLM.from_pretrained('qilowoq/AbLang_light')
# 중쇄(heavy chain) 모델
mlm_heavy_chain_model = AutoModelForMaskedLM.from_pretrained('qilowoq/AbLang_heavy')


In [ ]:
# 모델의 파라미터 수와 구조를 살펴봅니다.
n_params = sum(p.numel() for p in mlm_heavy_chain_model.parameters())
print(f'AbLang 모델은 {n_params}개의 학습 가능한 파라미터를 가집니다. \n')
mlm_heavy_chain_model


#### 2.2.2 예시 항체 서열

다음으로 설계에 사용할 중쇄와 경쇄를 고릅니다. 이들은 HuggingFace의 AbLang 예시 페이지에서 가져온 것입니다.


In [ ]:
# 중쇄와 경쇄의 가변 영역을 사용합니다.

heavy_chain_example =  'EVQLQESGPGLVKPSETLSLTCTVSGGPINNAYWTWIRQPPGKGLEYLGYVYHTGVTNYNPSLKSRLTITIDTSRKQLSLSLKFVTAADSAVYYCAREWAEDGDFGNAFHVWGQGTMVAVSSASTKGPSVFPLAPSSKSTSGGTAALGCL'
light_chain_example = 'GSELTQDPAVSVALGQTVRITCQGDSLRNYYASWYQQKPRQAPVLVFYGKNNRPSGIPDRFSGSSSGNTASLTISGAQAEDEADYYCNSRDSSSNHLVFGGGTKLTVLSQ'


#### 2.2.3 서열 마스킹하기

이 접근법의 핵심 매개변수 중 하나는 어떤 잔기를 마스킹하고 재설계할지 정하는 것입니다. 먼저, 임의의 서열에 임의의 지점에서 마스킹 절차를 적용할 수 있도록 재현 가능한(reproducible) 코드를 마련하는 것부터 시작합시다.


In [ ]:
# 서열 마스킹 편의 함수
def mask_seq_pos(sequence: str,
                      idx: int,
                      mask='[MASK]'):
    '''임의의 항체 서열과 서열 인덱스가 주어지면,
    해당 인덱스의 잔기를 마스크 토큰으로 바꿉니다.
    '''
    cleaned_sequence = sequence.replace(' ', '')  # 불필요한 공백 제거
    assert abs(idx) < len(sequence), "0부터 시작하는 인덱스 값은 (서열 길이 - 1)보다 작아야 합니다."
    cleaned_sequence = list(cleaned_sequence)  # 서열을 리스트로 변환
    cleaned_sequence[idx] = '*'                 # idx 위치를 마스킹
    masked_sequence = ' '.join(cleaned_sequence) # 리스트 -> 서열(공백 구분)
    masked_sequence = masked_sequence.replace('*', mask)
    return masked_sequence

# 테스트
assert mask_seq_pos('CAT', 1)=='C [MASK] T'

#TODO: pytest로 단위 테스트를 추가하여 assert가 동작하는지 확인할 수 있습니다


#### 2.2.3 모델 추론(Inference)


In [ ]:
### 1단계. 경쇄(light_chain) 서열을 마스킹합니다
mask_idx = 9
masked_light_chain = mask_seq_pos(light_chain_example, idx=mask_idx)
### 2단계. 토큰화
tokenized_input = tokenizer(masked_light_chain, return_tensors='pt')
### 3단계. 경쇄 모델에 통과
mlm_output = mlm_light_chain_model(**tokenized_input)
### 4단계. 출력을 디코딩하여 모델이 무엇을 채웠는지 확인
decoded_outs = tokenizer.decode(mlm_output.logits.squeeze().argmax(dim=1), skip_special_tokens=True)
print(f'모델 예측: 인덱스 {mask_idx} 위치에 {decoded_outs.replace(" ", "")[9]}')
print(f'예측된 서열: {decoded_outs.replace(" ", "")}')
print(f'시작 서열  : {light_chain_example}')


### 2.2.4 HuggingFace 파이프라인(Pipeline) 객체

잠깐, 마스크 토큰이 하나뿐인 토큰화 입력을 넣었으니 서열에서 단 한 곳만 바뀔 것으로 기대했을 텐데, 돌아온 결과는 넣은 것과 꽤 많이 다릅니다. 다행히 HuggingFace 소프트웨어에는 이를 해결할 수 있는 것이 있습니다: 파이프라인(Pipeline)입니다.

HuggingFace 파이프라인:
1. 파이프라인 객체는 추론(inference)을 감싸는 래퍼(wrapper)로, API 호출처럼 다룰 수 있습니다.
2. 우리가 쓸 수 있는 fill-mask 파이프라인은, 입력에서 하나의 마스크 토큰을 받아 그 서열의 점수(score), 채워진 토큰, 그리고 복원된 전체 서열을 담은 딕셔너리를 출력합니다.


실제로 동작하는 모습을 봅시다:


In [ ]:
from transformers import pipeline
filler = pipeline(task='fill-mask', model=mlm_light_chain_model, tokenizer=tokenizer)
filler(masked_light_chain) # 마스크 채우기


면책 조항: 더 철저한 항체 (재)설계를 위해서는 보통 Hie 등의 연구처럼, 서열의 모든 지점을 돌연변이시키고 전체 서열을 모아 점수를 매긴 뒤 상위 100개 정도의 항체를 발현시켜 검증하는 접근을 따르고자 할 것입니다. 직접 탐구해 보고 싶다면 도전 과제로 시도해 보세요!

또한 Hie 등의 실제 데이터를 참고하여, 예측된 항체 중 잘 작동하고 적합도를 높인 것이 있었는지 확인해 볼 수 있습니다.


### **2.3 한계점 (Limitations)**

유망하지만, 이 접근법에도 분명 단점이 있습니다. 주요 한계는 다음과 같습니다:

* 마스크 토큰이 1:1로 적용되므로 항체 길이가 고정됩니다(fixed length).
* 조건부 샘플링 단계에서 표적(target) 정보가 포함되지 않아, 서열 맥락에 따른 아미노산 선택에 영향을 주지 못합니다.
* 접근법이 단백질 언어 모델 선택에 민감합니다.

이 레터 [[18]](https://www.nature.com/articles/s41587-023-01991-6)는 Hie 등의 연구를 잘 요약하고 있으며, 이는 이 튜토리얼에서 제시한 방법에도 그대로 적용됩니다.


## 이 튜토리얼 인용하기

이 튜토리얼이 유용했다면, 아래와 같이 인용해 주시길 부탁드립니다:

```
@manual{Bioinformatics,
 title={An Introduction to Antibody Design Using Protein Language Models},
 organization={DeepChem},
 author={Karthikeyan, Dhuvarakesh and Menezes, Aaron},
 howpublished = {\url{https://github.com/deepchem/deepchem/blob/master/examples/tutorials/DeepChem_AntibodyTutorial_Simplified.ipynb}},
 year={2024},
}
```


## 참고 문헌(Works Cited)

[1] Hie, B.L., Shanker, V.R., Xu, D. et al. Efficient evolution of human antibodies from general protein language models. Nat Biotechnol 42, 275–283 (2024). https://doi.org/10.1038/s41587-023-01763-2

[2] Bretscher, P., & Cohn, M. (1970). A Theory of Self-Nonself Discrimination. Science, 169(3950), 1042–1049. doi:10.1126/science.169.3950.1042

[3] Cohn, M. The common sense of the self-nonself discrimination. Springer Semin Immun 27, 3–17 (2005). https://doi.org/10.1007/s00281-005-0199-1

[4] Gonzalez S, González-Rodríguez AP, Suárez-Álvarez B, López-Soto A, Huergo-Zapico L, Lopez-Larrea C. Conceptual aspects of self and nonself discrimination. Self Nonself. 2011 Jan;2(1):19-25. doi: 10.4161/self.2.1.15094. Epub 2011 Jan 1. PMID: 21776331; PMCID: PMC3136900.

[5] ROB J. DE BOER, PAULINE HOGEWEG, Self-Nonself Discrimination due to Immunological Nonlinearities: the Analysis of a Series of Models by Numerical Methods, Mathematical Medicine and Biology: A Journal of the IMA, Volume 4, Issue 1, 1987, Pages 1–32, https://doi.org/10.1093/imammb/4.1.1

[6] Cohn, M. A biological context for the self-nonself discrimination and the regulation of effector class by the immune system. Immunol Res 31, 133–150 (2005). https://doi.org/10.1385/IR:31:2:133

[7] Janeway CA Jr, Travers P, Walport M, et al. Immunobiology: The Immune System in Health and Disease. 5th edition. New York: Garland Science; 2001. Principles of innate and adaptive immunity. Available from: https://www.ncbi.nlm.nih.gov/books/NBK27090/

[8] Perelson, A. Modelling viral and immune system dynamics. Nat Rev Immunol. 2. , 28–36 (2002). https://doi.org/10.1038/nri700

[9] Shittu, A. (n.d.). Understanding immunological memory. ASM.org. https://asm.org/articles/2023/may/understanding-immunological-memory

[10] Janeway CA Jr, Travers P, Walport M, et al. Immunobiology: The Immune System in Health and Disease. 5th edition. New York: Garland Science; 2001. Chapter 8, T Cell-Mediated Immunity. Available from: https://www.ncbi.nlm.nih.gov/books/NBK10762/

[11] Glick, B., Chang, T. S., & Jaap, R. G. (1956). The Bursa of Fabricius and Antibody Production. Poultry Science, 35(1), 224–225. doi:10.3382/ps.0350224

[12] Nooren, I. M. (2003). NEW EMBO MEMBER’S REVIEW: Diversity of protein-protein interactions. EMBO Journal, 22(14), 3486–3492. https://doi.org/10.1093/emboj/cdg359

[13] Karolis Martinkus, Jan Ludwiczak, Kyunghyun Cho, Wei-Ching Liang, Julien Lafrance-Vanasse, Isidro Hotzel, Arvind Rajpal, Yan Wu, Richard Bonneau, Vladimir Gligorĳevic, & Andreas Loukas. (2024). AbDiffuser: Full-Atom Generation of in vitro Functioning Antibodies.

[14] Baris E. Suzek, Hongzhan Huang, Peter McGarvey, Raja Mazumder, Cathy H. Wu, UniRef: comprehensive and non-redundant UniProt reference clusters, Bioinformatics, Volume 23, Issue 10, May 2007, Pages 1282–1288, https://doi.org/10.1093/bioinformatics/btm098

[15] Tobias H Olsen, Iain H Moal, Charlotte M Deane, AbLang: an antibody language model for completing antibody sequences, Bioinformatics Advances, Volume 2, Issue 1, 2022, vbac046, https://doi.org/10.1093/bioadv/vbac046

[16] Yinhan Liu, Myle Ott, Naman Goyal, Jingfei Du, Mandar Joshi, Danqi Chen, Omer Levy, Mike Lewis, Luke Zettlemoyer, & Veselin Stoyanov. (2019). RoBERTa: A Robustly Optimized BERT Pretraining Approach.

[17] Olsen TH, Boyles F, Deane CM. Observed Antibody Space: A diverse database of cleaned, annotated, and translated unpaired and paired antibody sequences. Protein Sci. 2022 Jan;31(1):141-146. doi: 10.1002/pro.4205. Epub 2021 Oct 29. PMID: 34655133; PMCID: PMC8740823.

[18] Outeiral, C., Deane, C.M. Perfecting antibodies with language models.
Nat Biotechnol 42, 185–186 (2024). https://doi.org/10.1038/s41587-023-01991-6


